# Aula 12 · Lab 3 — API FastAPI: /query, /health, /metrics

> **Objetivo:** Implementar e testar os endpoints de produção da API RAG  
> com autenticação por API Key, validação Pydantic e métricas de uso.

## O que você vai construir

| Endpoint | Método | Descrição |
|----------|--------|-----------|
| `/query` | POST | Consulta RAG com contexto e resposta gerada |
| `/health` | GET | Status de todos os componentes |
| `/metrics` | GET | Latência P50/P95, tokens, erros |

---
**Referência:** FastAPI Documentation (2024). https://fastapi.tiangolo.com

In [ ]:
!pip install -q fastapi uvicorn httpx pydantic openai nest-asyncio
import nest_asyncio
nest_asyncio.apply()
print("✅ Pronto")

In [ ]:
# ============================================================
# CÓDIGO COMPLETO DA API — Salvo em /tmp/rag_app/app.py
# ============================================================

import os
os.makedirs("/tmp/rag_app", exist_ok=True)

APP_CODE = '''
from fastapi import FastAPI, HTTPException, Depends, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import APIKeyHeader
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager
from typing import Optional, List
import asyncio, time, os, logging, numpy as np

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("rag_api")

# --- Autenticação ---
API_KEYS = set(os.environ.get("API_KEYS", "dev-key-123,test-key-456").split(","))
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

def verificar_key(key: str = Depends(api_key_header)):
    if key not in API_KEYS:
        raise HTTPException(403, "API Key inválida")
    return key

# --- Modelos ---
class QueryRequest(BaseModel):
    pergunta: str = Field(..., min_length=5, max_length=2000)
    filtro_tipo: Optional[str] = None
    top_k: int = Field(5, ge=1, le=20)

class FonteDoc(BaseModel):
    titulo: str
    fonte: str
    score: float

class QueryResponse(BaseModel):
    resposta: str
    fontes: List[FonteDoc]
    latencia_ms: float
    tokens_usados: int
    modelo: str

class HealthResponse(BaseModel):
    status: str
    versao: str
    opensearch_ok: bool
    langfuse_ok: bool
    uptime_s: float

class MetricsResponse(BaseModel):
    total_queries: int
    latencia_media_ms: float
    latencia_p50_ms: float
    latencia_p95_ms: float
    tokens_totais: int
    erros_total: int
    custo_estimado_usd: float

# --- Estado da aplicação ---
class Estado:
    inicio = time.time()
    queries = 0
    latencias: List[float] = []
    tokens = 0
    erros = 0
    # gpt-4o-mini: $0.15/1M input, $0.60/1M output (aproximado)
    CUSTO_POR_TOKEN = 0.000_000_6

    def registrar(self, lat_ms, n_tokens):
        self.queries += 1
        self.latencias = (self.latencias + [lat_ms])[-1000:]
        self.tokens += n_tokens

    def percentil(self, p):
        if not self.latencias: return 0.0
        return float(np.percentile(self.latencias, p))

estado = Estado()


@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info("🚀 API RAG iniciando")
    yield
    logger.info("⛔ API RAG encerrando")

app = FastAPI(
    title="RAG Jurídico API",
    description="API RAG de produção para direito e segurança pública",
    version="1.0.0",
    lifespan=lifespan
)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


@app.post("/query", response_model=QueryResponse, summary="Consulta RAG")
async def query_rag(req: QueryRequest, key: str = Depends(verificar_key)):
    """Executa o pipeline RAG e retorna resposta com fontes e latência."""
    inicio = time.time()
    try:
        # ============================================================
        # SUBSTITUA ESTE BLOCO PELA CHAMADA AO SEU pipeline_rag()
        # ============================================================
        await asyncio.sleep(0.05)  # Simula processamento
        resposta_mock = f"Resposta simulada para: {req.pergunta[:50]}"
        fontes_mock = [{"titulo": "Doc exemplo", "fonte": "Lei mock", "score": 0.95}]
        tokens_mock = 180
        # ============================================================

        lat = (time.time() - inicio) * 1000
        estado.registrar(lat, tokens_mock)
        logger.info(f"Query OK | lat={lat:.0f}ms | tokens={tokens_mock}")

        return QueryResponse(
            resposta=resposta_mock,
            fontes=[FonteDoc(**f) for f in fontes_mock],
            latencia_ms=round(lat, 2),
            tokens_usados=tokens_mock,
            modelo="gpt-4o-mini"
        )
    except Exception as e:
        estado.erros += 1
        logger.error(f"Erro em /query: {e}")
        raise HTTPException(500, detail=str(e))


@app.get("/health", response_model=HealthResponse, summary="Health check")
async def health():
    """Verifica status de todos os componentes."""
    # Em produção: teste conexões reais
    opensearch_ok = True  # os_client.ping()
    langfuse_ok   = True  # langfuse.auth_check()

    return HealthResponse(
        status="healthy" if (opensearch_ok and langfuse_ok) else "degraded",
        versao="1.0.0",
        opensearch_ok=opensearch_ok,
        langfuse_ok=langfuse_ok,
        uptime_s=round(time.time() - estado.inicio, 1)
    )


@app.get("/metrics", response_model=MetricsResponse, summary="Métricas de uso")
async def metrics(key: str = Depends(verificar_key)):
    """Retorna métricas detalhadas de uso e performance."""
    lat = estado.latencias
    return MetricsResponse(
        total_queries=estado.queries,
        latencia_media_ms=round(sum(lat)/len(lat), 2) if lat else 0.0,
        latencia_p50_ms=round(estado.percentil(50), 2),
        latencia_p95_ms=round(estado.percentil(95), 2),
        tokens_totais=estado.tokens,
        erros_total=estado.erros,
        custo_estimado_usd=round(estado.tokens * Estado.CUSTO_POR_TOKEN, 4)
    )
'''

with open("/tmp/rag_app/app.py", "w") as f:
    f.write(APP_CODE.strip())

# requirements.txt
with open("/tmp/rag_app/requirements.txt", "w") as f:
    f.write("fastapi>=0.111.0\nuvicorn[standard]>=0.30.0\nhttpx>=0.27.0\npydantic>=2.7.0\nnumpy>=1.26.0\n")

print("✅ Arquivos criados em /tmp/rag_app/")

In [ ]:
# ============================================================
# INICIANDO SERVIDOR E TESTANDO ENDPOINTS
# ============================================================

import subprocess, time, httpx

proc = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8001", "--log-level", "warning"],
    cwd="/tmp/rag_app",
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)

BASE = "http://localhost:8001"
KEY  = {"X-API-Key": "dev-key-123"}

resultados_testes = []

print("" + "="*55)
print("SUITE DE TESTES DA API RAG")
print("="*55)

# Teste 1: Health (sem auth)
r = httpx.get(f"{BASE}/health")
ok = r.status_code == 200
resultados_testes.append(("GET /health", ok, r.status_code))
print(f"\n[1] GET /health")
print(f"    Status: {r.status_code} | {'✅ OK' if ok else '❌ FALHOU'}")
print(f"    Body:   {r.json()}")

# Teste 2: Query válida
payload = {"pergunta": "O que é dado pessoal sensível segundo a LGPD?", "top_k": 5}
r = httpx.post(f"{BASE}/query", json=payload, headers=KEY)
ok = r.status_code == 200
resultados_testes.append(("POST /query", ok, r.status_code))
print(f"\n[2] POST /query")
print(f"    Status: {r.status_code} | {'✅ OK' if ok else '❌ FALHOU'}")
dados = r.json()
print(f"    Resposta: {dados['resposta'][:80]}...")
print(f"    Latência: {dados['latencia_ms']}ms | Tokens: {dados['tokens_usados']}")

# Teste 3: Query sem auth
r = httpx.post(f"{BASE}/query", json=payload)
ok = r.status_code == 403
resultados_testes.append(("POST /query (sem auth)", ok, r.status_code))
print(f"\n[3] POST /query (sem autenticação)")
print(f"    Status: {r.status_code} | {'✅ Corretamente negado' if ok else '❌ Deveria ser 403'}")

# Teste 4: Query inválida (pergunta muito curta)
r = httpx.post(f"{BASE}/query", json={"pergunta": "oi"}, headers=KEY)
ok = r.status_code == 422
resultados_testes.append(("POST /query (validação)", ok, r.status_code))
print(f"\n[4] POST /query (pergunta muito curta — validação Pydantic)")
print(f"    Status: {r.status_code} | {'✅ Validação funcionou' if ok else '❌ Deveria ser 422'}")

# Teste 5: Metrics
r = httpx.get(f"{BASE}/metrics", headers=KEY)
ok = r.status_code == 200
resultados_testes.append(("GET /metrics", ok, r.status_code))
print(f"\n[5] GET /metrics")
print(f"    Status: {r.status_code} | {'✅ OK' if ok else '❌ FALHOU'}")
m = r.json()
print(f"    Queries: {m['total_queries']} | P95: {m['latencia_p95_ms']}ms | Custo: ${m['custo_estimado_usd']}")

# Sumário
print("\n" + "="*55)
print("SUMÁRIO")
print("="*55)
passaram = sum(1 for _, ok, _ in resultados_testes if ok)
print(f"Testes: {passaram}/{len(resultados_testes)} passaram")
for nome, ok, status in resultados_testes:
    print(f"  {'✅' if ok else '❌'} {nome} ({status})")

proc.terminate()
print("\nServidor encerrado")

In [ ]:
# ============================================================
# BÔNUS: Documentação automática (Swagger UI)
# ============================================================
# O FastAPI gera documentação automática.
# Em produção, acesse: http://seu-host:8000/docs
# Para ver a documentação local, inicie o servidor e acesse:
#   http://localhost:8001/docs  (Swagger UI)
#   http://localhost:8001/redoc (ReDoc)

print("📖 FastAPI gera documentação automática a partir dos tipos Pydantic e docstrings.")
print("\nEndpoints de documentação:")
print("  http://localhost:8001/docs    (Swagger UI — interativo)")
print("  http://localhost:8001/redoc   (ReDoc — navegável)")
print("  http://localhost:8001/openapi.json (Schema OpenAPI 3.0)")

print("\n💡 Dica para o Projeto Final:")
print("   Inclua a URL do Swagger na sua defesa técnica.")
print("   Demonstre a API ao vivo usando o Swagger UI.")